# Neuron count comparison pytorch

Original experiment from [Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow](https://github.com/Muhammad-Huzifa/Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow/blob/dfa17b2b298631b10e5904b2f1c39ad64d424bc1/Pytorch/ANN%20using%20Optuna%20Different%20Neurons.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)

Download and extract the dataset described in `docs/DATASETS.md` before running the model cells. Notebook-specific credential uploads and mounted-drive setup have been removed.

In [ ]:
! pip install opendatasets

Download and extract the dataset described in `docs/DATASETS.md` before running the model cells. Notebook-specific credential uploads and mounted-drive setup have been removed.

In [ ]:
data = pd.read_csv(str(TABULAR / 'fashion-mnist_train.csv'))

In [ ]:
data.head()


In [ ]:
# Create a 4x4 grid of images
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle("First 16 Images", fontsize=16)

# Plot the first 16 images from the dataset
for i, ax in enumerate(axes.flat):
    img = data.iloc[i, 1:].values.reshape(28, 28)  # Reshape to 28x28
    ax.imshow(img)  # Display in grayscale
    ax.axis('off')  # Remove axis for a cleaner look
    ax.set_title(f"Label: {data.iloc[i, 0]}")  # Show the label

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to fit the title
plt.show()


In [ ]:
# train test split

X = data.iloc[:, 1:].values
y = data.iloc[:, 0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    # Convert to PyTorch tensors
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
class MyNN(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()
        layers = []

        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer[i]))
            layers.append(nn.BatchNorm1d(neurons_per_layer[i]))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate[i]))
            input_dim = neurons_per_layer[i]  # Update input for next layer

        layers.append(nn.Linear(neurons_per_layer[-1], output_dim))  # Output layer

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
best_params = {
    'num_hidden_layers': 1,
    'layer0_nodes': 56,
    'dropout_layer0': 0.2,
    'epochs': 30,
    'learning_rate': 0.00036392673311859056,
    'batch_size': 16,
    'optimizer': 'RMSprop',
    'weight_decay': 8.13662957205625e-05
}

# Initialize model with best parameters
input_dim = 784  # Assuming input is 28x28 images (e.g., MNIST dataset)
output_dim = 10  # Assuming 10 classes

num_hidden_layers = best_params['num_hidden_layers']
neurons_per_layer = [best_params['layer0_nodes']]
dropout_rates = [best_params['dropout_layer0']]

model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rates)
# model.to(device)

# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])



In [ ]:
# Training Loop
epochs = best_params['epochs']
batch_size = best_params['batch_size']

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

for epoch in range(epochs):
    model.train()
    for batch_features, batch_labels in train_loader:
        # batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        optimizer.zero_grad()
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()

# Evaluation
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
class Parent:
  def __init__(self) -> None:
      print('working')

class Child(Parent):
  def __init__(self) -> None:
      super().__init__()
      print('How')


In [ ]:
new = Child()